# HeritAgent — EXP-0001 koşusu (Colab, A100 80 GB)

Bu notebook asıl deneyi çalıştırır (ADR-0007, ADR-0008). **Önce şunlar tamam olmalı:**
1. koşunun kodu, modeli ve worker sayısı için `EXP-0001-run-rehearsal.ipynb` CPU'da geçti: koşunun tamamını, geçidin altı turunu, arızaları ve iki geri yüklemeyi taklit bir sunucuya karşı oynar (geçmediyse bu notebook ilk dakikalarda durur, `run` da başlamaz);
2. pilot notebook'u çalıştı, `pilot-report.json` incelendi ve gerekiyorsa ADR-0011 D6 kararını verdiniz;
3. hesap tavanı kararı verildi;
4. Critic onayı (artık benim görevim) verildi.

**Nasıl çalışır:**
- Koşu birkaç oturuma yayılabilir. Her oturumda "Tümünü çalıştır" deyin; aynı `RUN_ID` ile kaldığı yerden devam eder. Kaydedilmiş bir bölüm asla yeniden oynanmaz.
- Koşu, başladığı commit ve model revizyonuyla devam eder; bunlar Drive'daki kayıttan okunur.
- Koşu yerel diskte çalışır ve kopyası her dakika Drive'a alınır (`heritagent/runs/<RUN_ID>/mirror`). Oturum koparsa en son kopyadan devam eder; kopyadan sonraki birkaç dakikalık iş yeniden oynanır.
- `STOP_AFTER_HOURS`: bu oturumda en fazla kaç saat çalışılacağı. Süre dolunca yeni bölüm başlamaz, süren bölümler bitip kaydedilir, hücre temiz durur ve son hücre GPU'yu bırakır. CU bütçenizi böyle bölebilirsiniz.
- 96 saatlik tavan yalnızca çalışılan süreyi sayar (ADR-0008 D2). 14.500 bölüm ve 507.500 çağrı tavanları da geçerlidir (ADR-0012 D9); her geçit turundan önce koşu, turun bölümleri ve son değerlendirme (1.358 bölüm) için yer olup olmadığına bakar. Tavanda ya da yer kalmadığında koşu "incomplete" olarak durur.
- **Aynı koşuyu asla iki sekmede ya da iki oturumda birden çalıştırmayın.** İkincisi birincisini durdurur.

**Tahmini maliyet (M):** pilot projeksiyonuna bakın. Örneğin saatte 150 bölümle 5000 bölüm yaklaşık 33 saat eder; saatte 7.5 CU ile yaklaşık 250 CU. Buna her oturumun kurulumu (~30–50 dk) ve miner'ın çağrıları eklenir. Kurulum süresi ve bir kopmadan sonra yeniden oynanan iş 96 saatlik tavana sayılmaz, ama CU harcar.

In [ ]:
# Ayarlar
RUN_ID = "EXP-0001-run-1"  # yeni bir koşu için yeni ad; devam için aynı ad
REPO_URL = "https://github.com/sametyilmaztemel/heritagent.git"
CODE_REF = "main"          # yalnızca YENİ koşuda; devamda koşunun kendi commit'i kullanılır
DRIVE_ROOT = "/content/drive/MyDrive/heritagent"
WORKERS = 16               # pilotla aynı değer önerilir
MODEL = "Qwen/Qwen3.8-27B"  # pilotla aynı model
STOP_AFTER_HOURS = 6.0     # bu oturumda en fazla bu kadar saat; None = bitene kadar
AUTO_RELEASE = True        # oturum bitince ya da bir hata olursa GPU'yu bırak (CU harcamasını durdurur)

In [ ]:
# 0) Bir hücre hata verirse "Tümünü çalıştır" orada durur. AUTO_RELEASE açıksa sunucu günlüğü
# Drive'a kopyalanır, Drive boşaltılır ve GPU bırakılır: hatadan sonra boşta bekleyen GPU da CU
# harcar. Bir deney komutu hâlâ çalışıyorsa ya da son kopyası Drive'a alınamamış bir koşu
# varsa GPU BIRAKILMAZ (o iş yalnızca bu makinededir).
import glob, os, shutil, time
from google.colab import drive, runtime as colab_runtime  # kurulumdan önce yüklenir


def unmirrored():
    """Son kopyası Drive'a alınamamış yerel koşu klasörleri (ADR-0008 D5)."""
    return sorted(os.path.dirname(p) for p in glob.glob("/content/*/UNMIRRORED.json"))


def running_commands():
    """Bu makinede hâlâ çalışan run/pilot/snapshot/rehearse komutlarının süreç numaraları. Çekirdek
    yeniden başladıysa önceki komut kendiliğinden durur, ama süren bölümlerini kaydedip son
    kopyayı alana dek (birkaç dakika) çalışır."""
    found = []
    for entry in os.listdir("/proc"):
        if not entry.isdigit() or int(entry) == os.getpid():
            continue
        try:
            with open(f"/proc/{entry}/cmdline", "rb") as handle:
                argv = handle.read().split(b"\0")
        except OSError:
            continue  # bu arada bitti
        if b"evaluation.exp0001.cli" in argv:
            command = argv[argv.index(b"evaluation.exp0001.cli") + 1:][:1]
            if command and command[0] in (b"run", b"pilot", b"snapshot", b"rehearse"):
                found.append(int(entry))
    return sorted(found)


def release():
    """Drive'ı boşaltır ve GPU'yu bırakır; bir komut hâlâ çalışıyorsa ya da aynalanmamış bir
    koşu varsa bırakmaz."""
    busy = running_commands()
    if busy:
        print("GPU BIRAKILMADI: bir deney komutu hâlâ çalışıyor (süreç", busy, "). Çekirdek "
              "yeniden başladıysa bu önceki komuttur: süren bölümlerini kaydedip son kopyayı "
              "alıyor.")
        print("Bir koşu, pilot ya da snapshot ise: birkaç dakika bekleyin, sonra o hücreyi yeniden "
              "çalıştırın (kaldığı yerden sürer) ya da release() çalıştırın. 15 dakikada bitmezse: "
              "import signal; os.killpg(<süreç>, signal.SIGTERM) ile sonlandırın; son kopyadan "
              "sonraki kayıtlar bu makinede kalır (UNMIRRORED.json): ardından cli('snapshot', "
              "'--run-dir', <klasör>, '--mirror-dir', <ayna>) çalıştırın (ayna: o koşunun ya da "
              "pilotun Drive'daki mirror klasörü). Runtime > Disconnect and delete runtime ise o "
              "kayıtları siler.")
        print("Bir prova (rehearse) ise: kaldığı yerden sürmez ve işaret bırakmaz. Birkaç dakika "
              "bekleyin (15 dakikada bitmezse os.killpg(<süreç>, signal.SIGTERM)), sonra "
              "release() çalıştırın ya da prova hücresini yeniden çalıştırıp yeni bir prova "
              "başlatın.")
        return
    kept = unmirrored()
    if kept:
        print("GPU BIRAKILMADI: son kopyası Drive'a alınamamış iş var:", kept)
        print("1) Drive'ı yeniden bağlayın: drive.mount('/content/drive', force_remount=True)")
        print("2) Her klasör için: cli('snapshot', '--run-dir', <klasör>, '--mirror-dir', <ayna>)")
        print("   (ayna: koşu için RUN_DRIVE + '/mirror', pilot için PILOT_DRIVE + '/mirror')")
        print("3) Kalan hücreleri (durum, rapor kopyası) çalıştırın; sonra release() çalıştırın: "
              "Drive'ı boşaltır ve GPU'yu bırakır.")
        return
    try:  # Drive'a yazılanlar makine kapanmadan yüklensin; en fazla 10 dakika beklenir
        drive.flush_and_unmount(timeout_ms=600_000)
    except BaseException as exc:  # bağlanmamış, takılmış ya da durdurulmuş: GPU yine bırakılır
        print("Drive boşaltılamadı:", repr(exc))
    print("Çalışma zamanı (GPU) bırakılıyor.")
    try:
        colab_runtime.unassign()
    except Exception as exc:
        print("GPU bırakılamadı; bağlantıyı kendiniz kesin:", exc)


def _on_error(shell, etype, value, tb, tb_offset=None):
    try:
        shell.showtraceback((etype, value, tb), tb_offset=tb_offset)
        if not AUTO_RELEASE:
            print("AUTO_RELEASE = False: GPU bağlı kalıyor; işiniz bitince bağlantıyı kesin.")
            return
        try:
            if os.path.isdir(DRIVE_ROOT) and os.path.exists("/content/vllm-server.log"):
                saved = f"{DRIVE_ROOT}/logs/vllm-server-{time.strftime('%Y%m%d-%H%M%S')}.log"
                os.makedirs(os.path.dirname(saved), exist_ok=True)
                shutil.copy("/content/vllm-server.log", saved)
                print("Sunucu günlüğü Drive'da:", saved)
        except Exception as exc:
            print("Sunucu günlüğü Drive'a kopyalanamadı:", exc)
        if etype.__name__ != "Bitti":
            print("Bir adım başarısız oldu. Hata metnini bana gönderin.")
    finally:  # günlükteki ya da Drive'daki bir hata GPU'nun bırakılmasını engellemez
        if AUTO_RELEASE:
            release()


get_ipython().set_custom_exc((Exception,), _on_error)

In [ ]:
# 1) GPU: yalnızca tek bir A100 80 GB (ADR-0008 D1). Başka bir GPU'da hiçbir şey kurulmaz.
import re, subprocess
query = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                        "--format=csv,noheader,nounits"], capture_output=True, text=True)
print(query.stdout or query.stderr)
gpus = [line.split(", ") for line in query.stdout.strip().splitlines()]
assert len(gpus) == 1 and "A100" in gpus[0][0] and int(float(gpus[0][1])) >= 76000, (
    "Bu notebook yalnızca TEK bir A100 80 GB ile çalışır. Çalışma zamanı > Çalışma zamanı "
    "türünü değiştir > A100 GPU seçin ve yeniden bağlanın.")
header = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
CUDA_DRIVER = float(re.search(r"CUDA Version:\s*([0-9.]+)", header).group(1))
print("Sürücünün desteklediği en yüksek CUDA:", CUDA_DRIVER)
assert CUDA_DRIVER >= 12.0, "Sürücü CUDA 12'den eski; kurulum yapılamaz. Bana sürücü sürümünü yazın."

In [ ]:
# 2) Google Drive: kalıcı kayıtlar burada durur (oturum kapanınca yerel disk silinir).
import os
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)
RUN_DRIVE = f"{DRIVE_ROOT}/runs/{RUN_ID}"
DONE = f"{RUN_DRIVE}/report.json"


class Bitti(Exception):
    pass


if os.path.exists(DONE):  # iş zaten bitmiş: kurulum ve GPU harcaması yok
    print(open(DONE).read()[:3000])
    raise Bitti("Koşu zaten bitti; rapor: " + DONE + " (Çalışma zamanı > Bağlantıyı kes ve sil)")

In [ ]:
# 2b) Model revizyonu: pilotun sabitlediği revizyon.
revision_file = f"{DRIVE_ROOT}/model-revision-{MODEL.replace('/', '--')}.txt"
assert os.path.exists(revision_file), "Önce pilot notebook'unu çalıştırın: model revizyonu orada sabitlenir."
MODEL_REVISION = revision = open(revision_file).read().strip()
print("Model revizyonu:", MODEL_REVISION)

In [ ]:
# 3) Kod: depoyu klonla ve CODE_REF noktasına geç.
import os, subprocess
REPO = "/content/heritagent"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--quiet", REPO_URL, REPO], check=True)
subprocess.run(["git", "-C", REPO, "fetch", "--quiet", "origin"], check=True)


def checkout(ref):
    target = f"origin/{ref}" if subprocess.run(
        ["git", "-C", REPO, "rev-parse", "--verify", "--quiet", f"origin/{ref}"],
        capture_output=True).returncode == 0 else ref
    subprocess.run(["git", "-C", REPO, "checkout", "--quiet", "--detach", target], check=True)
    return subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True,
                          text=True, check=True).stdout.strip()


COMMIT = checkout(CODE_REF)
os.chdir(REPO)
print("Kod commit'i:", COMMIT)

In [ ]:
# 3b) Devam eden bir koşu kendi commit'iyle sürer: commit Drive'daki run-start kaydından okunur.
# Bunun için önce yalnızca deponun kendisi kurulur (saniyeler); ağır kurulum, koşunun commit'ine
# geçildikten ve o commit'in provası bulunduktan sonra yapılır.
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", REPO], check=True)
found = subprocess.run([sys.executable, "-m", "evaluation.exp0001.cli", "run-commit",
                        "--mirror-dir", f"{RUN_DRIVE}/mirror"], capture_output=True, text=True)
print(found.stderr[-3000:], end="")
assert found.returncode == 0, "run-commit başarısız oldu (çıktı yukarıda)."
run_commit = found.stdout.strip()
if run_commit and run_commit != COMMIT:
    COMMIT = checkout(run_commit)
print("Koşunun commit'i:", COMMIT, "(devam)" if run_commit else "(yeni koşu)")

In [ ]:
# 3c) GPU'suz koşu provası: bu kodun ve modelin koşu provası CPU'da geçmiş olmalı. Geçmediyse
# durulur ve GPU bırakılır (birkaç dakikalık CU): önce EXP-0001-run-rehearsal.ipynb'yi CPU
# çalışma zamanında çalıştırın.
PASSED = f"{DRIVE_ROOT}/rehearsal/run-passed-{COMMIT}-{MODEL.replace('/', '--')}.json"
if not os.path.exists(PASSED):
    import glob
    if run_commit:  # devam eden bir koşu kendi commit'iyle sürer: CODE_REF'in etkisi yok
        raise RuntimeError("Bu koşu kendi commit'iyle (" + COMMIT + ") sürer; CODE_REF burada "
                           "etkisizdir. Bu commit ve " + MODEL + " için koşu provası yok: "
                           "EXP-0001-run-rehearsal.ipynb'yi CPU çalışma zamanında CODE_REF = \""
                           + COMMIT + "\", koşunun modeli (MODEL) ve WORKERS ile çalıştırın.")
    passed = sorted(os.path.basename(p)[len("run-passed-"):][:40] for p in glob.glob(
        f"{DRIVE_ROOT}/rehearsal/run-passed-*-{MODEL.replace('/', '--')}.json"))
    raise RuntimeError("Bu kodun (commit " + COMMIT[:12] + ") ve " + MODEL + " modelinin koşu "
                       "provası yok. Önce EXP-0001-run-rehearsal.ipynb'yi CPU çalışma zamanında "
                       "çalıştırın (aynı CODE_REF, MODEL ve WORKERS ile)." + (
                           " Bu model için koşu provası geçmiş kodlar: " + ", ".join(passed) + ". O "
                           "kodu çalıştırmak için CODE_REF'e birini yazın." if passed else ""))
print("GPU'suz koşu provası geçmiş:", PASSED)


In [ ]:
# 4) Kurulum (~5-10 dk): deneyin komutları Colab'ın kendi Python'unu ve paketlerini kullanmaz.
# Her oturumda depodaki environments/exp0001-colab/ dosyalarına göre (CODE_REF'teki hâlleriyle)
# AYNI ortam kurulur: hash'iyle uv (uv-requirements.txt), sabit Python (python-version), bütün
# paketlerin hash'li kilidi (requirements.lock: vLLM 0.29.0, torch 2.13.0 CUDA 12.9 yapısı,
# transformers, tokenizers, huggingface-hub, xgrammar, llguidance, jinja2, ALFWorld 0.4.2,
# TextWorld 1.7.0 ve bütün bağımlılıkları) ve kaynaktan derlenen iki paketin hash'li derleme
# araçları (build-constraints.txt). Prova ile koşu böylece aynı ortamda çalışır; sonraki hücre
# ortamı denetler. Koşu başladığı sürümleri kaydeder; başka sürümlerle devam etmez (ADR-0008 D4).
import hashlib, os, signal, subprocess, sys, time

ENV = f"{REPO}/environments/exp0001-colab"
VENV = "/content/heritagent-venv"
PY = f"{VENV}/bin/python"  # deneyin komutları bu ortamın Python'uyla çalışır


def command_environment():
    """Komutların ortam değişkenleri: çekirdeğin Python ve matplotlib ayarları (PYTHONPATH,
    MPLBACKEND gibi) bu ortama geçmez; ortamın bin klasörü PATH'te önce gelir (vllm serve)."""
    kept = {k: v for k, v in os.environ.items() if not k.startswith(("PYTHON", "MPL"))}
    return {**kept, "VIRTUAL_ENV": VENV, "PATH": f"{VENV}/bin:{os.environ['PATH']}",
            "MPLBACKEND": "Agg", "PYTHONNOUSERSITE": "1"}


def vllm_servers():
    """Bu makinede çalışan `vllm serve` süreçlerinin numaraları."""
    found = []
    for entry in os.listdir("/proc"):
        if not entry.isdigit():
            continue
        try:
            with open(f"/proc/{entry}/cmdline", "rb") as handle:
                argv = handle.read().split(b"\0")
        except OSError:
            continue  # bu arada bitti
        if b"serve" in argv[1:3] and any(os.path.basename(arg) == b"vllm" for arg in argv[:2]):
            found.append(int(entry))
    return sorted(found)


SERVER_STOP_SECONDS = (60, 30)  # SIGTERM'den sonra, SIGKILL'den sonra en fazla bu kadar beklenir
SERVER_INFO = "/content/vllm-server.json"  # serve komutunun sunucu kaydı


def stop_servers(pids):
    """Eski ortamdan kalan vLLM sunucularını süreç gruplarıyla (motor süreçleri de) durdurur;
    hiçbir deney komutu onları kullanmıyor. Yeni ortamın sunucusu, model yeniden yüklenerek
    sonraki hücrelerde açılır."""
    def send(pid, sig):
        for kill in (os.killpg, os.kill):  # sunucu kendi oturumunda başlar: grubu onunla aynı
            try:
                kill(pid, sig)
                return
            except ProcessLookupError:
                continue  # grup ya da süreç bu arada bitti

    left = list(pids)
    for sig, wait in zip((signal.SIGTERM, signal.SIGKILL), SERVER_STOP_SECONDS):
        for pid in left:
            send(pid, sig)
        deadline = time.monotonic() + wait
        while vllm_servers() and time.monotonic() < deadline:
            time.sleep(1)
        left = vllm_servers()
        if not left:
            break
    if left:
        raise RuntimeError(f"Eski ortamın vLLM sunucusu durmadı (süreç {left}); çalışma zamanı "
                           f"bırakılacak, yeni bir oturumda yeniden başlayın.")
    if os.path.exists(SERVER_INFO):  # durdurulan sunucunun kaydı: `serve` onu canlı sanmasın
        os.remove(SERVER_INFO)


def install():
    """Ortamı kurar. Bir deney komutu çalışırken hiçbir şeye dokunmaz; ortam değişecekse önce
    eski ortamdan kalan vLLM sunucularını durdurur (kurulan ortam, kurulduğu dosyaların
    damgasını taşır)."""
    busy = running_commands()
    if busy:  # yeniden başlayan bir çekirdek, hâlâ çalışan bir komutun ortamına dokunmaz
        raise RuntimeError(f"Bir deney komutu hâlâ çalışıyor (süreç {busy}); o bitince bu "
                           f"hücreyi ve sonrakileri yeniden çalıştırın.")
    with open(f"{ENV}/python-version", encoding="utf-8") as handle:
        python = handle.read().strip()
    digest = hashlib.sha256()
    for name in ("python-version", "uv-requirements.txt", "requirements.lock",
                 "build-constraints.txt"):
        with open(f"{ENV}/{name}", "rb") as handle:
            digest.update(handle.read())
    stamp = f"{VENV}/heritagent-environment.sha256"
    try:
        with open(stamp, encoding="utf-8") as handle:
            same = handle.read().strip() == digest.hexdigest()
    except OSError:
        same = False
    servers = vllm_servers()
    if servers and not same:
        stop_servers(servers)
    env = command_environment()  # kaynaktan derlenen paketler de çekirdeğin ayarlarını görmez
    uv = [sys.executable, "-m", "uv"]
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--require-hashes",
                    "--only-binary", ":all:", "-r", f"{ENV}/uv-requirements.txt"], check=True)
    try:
        with open(f"{VENV}/pyvenv.cfg", encoding="utf-8") as handle:
            made = dict(line.split(" = ", 1) for line in handle.read().splitlines()
                        if " = " in line)
    except OSError:
        made = {}
    if made.get("version_info") != python or not os.path.exists(PY):
        subprocess.run(uv + ["venv", "--clear", "--managed-python", "--python", python, VENV],
                       check=True, env=env)
    build = ["--build-constraints", f"{ENV}/build-constraints.txt"]
    subprocess.run(uv + ["pip", "sync", "--python", PY, "--require-hashes", "--index-strategy",
                         "unsafe-best-match", *build, f"{ENV}/requirements.lock"],
                   check=True, env=env)
    subprocess.run(uv + ["pip", "install", "--python", PY, "--no-deps", *build, "-e", REPO],
                   check=True, env=env)
    subprocess.run(uv + ["pip", "check", "--python", PY], check=True, env=env)  # bağımlılıklar
    with open(stamp, "w", encoding="utf-8") as handle:
        handle.write(digest.hexdigest() + "\n")


def cli(*args):
    """Deneyin bir komutunu çalıştırır ve çıktısını canlı yazar. Komut başarısız olursa hata
    verir; böylece "Tümünü çalıştır" sonraki (para harcayan) hücrelere geçmez.
    Durdur düğmesi: komut, süre sınırındaki gibi durur (sürmekte olan bölümler kaydedilir,
    son kopya Drive'a alınır); ikinci basış onu hemen sonlandırır."""
    command = [PY, "-u", "-m", "evaluation.exp0001.cli", *map(str, args)]
    # kendi süreç grubunda: durdurma sinyali ona yalnızca buradan, bir kez ulaşır; çekirdek
    # yeniden başlarsa ya da ölürse komut da aynı şekilde durur (HERITAGENT_STOP_WITH_PARENT)
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1, start_new_session=True,
                          env={**command_environment(),
                               "HERITAGENT_STOP_WITH_PARENT": str(os.getpid())}) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
        except KeyboardInterrupt:
            process.send_signal(signal.SIGINT)
            print("Durduruluyor: süren bölümler kaydediliyor ve son kopya alınıyor (birkaç "
                  "dakika sürebilir). Hemen sonlandırmak için yeniden basın.", flush=True)
            try:
                for line in process.stdout:
                    print(line, end="", flush=True)
            except KeyboardInterrupt:
                os.killpg(process.pid, signal.SIGTERM)
                print("Sonlandırıldı. Bu bir run, pilot ya da snapshot komutuysa son kopyadan "
                      "sonraki kayıtlar yalnızca bu makinede olabilir ve UNMIRRORED.json "
                      "bırakılmıştır (unmirrored() ile bakın): çalışma zamanı, cli('snapshot', "
                      "'--run-dir', <klasör>, '--mirror-dir', <ayna>) çalışana dek "
                      "bırakılmaz.", flush=True)
            raise
    if process.returncode == 3:  # cli.EXIT_UNMIRRORED
        raise RuntimeError(f"`cli {args[0]}`: son kopya Drive'a alınamadı (UNMIRRORED); en yeni "
                           f"kayıtlar yalnızca bu makinede. Çalışma zamanı bırakılmayacak.")
    if process.returncode != 0:
        raise RuntimeError(f"`cli {args[0]}` başarısız oldu (çıkış kodu {process.returncode}); "
                           f"sonraki hücreler çalıştırılmadı.")


install()  # en sonda: kurulum durursa da cli tanımlıdır (kurtarma adımları onu kullanır)


In [ ]:
# 4c) Ortam kilitli olanın aynısı mı (Python, her paket bir kez, vllm bu ortamdan; değilse
# veri ve model indirilmeden durulur), kurulan sürümler açılıyor mu, CUDA görünüyor mu?
cli("env-check", "--repo", REPO)
check = subprocess.run([PY, "-c", "import vllm, torch, matplotlib; assert torch.cuda.is_available(), "
                        "'CUDA görünmüyor'; print('vllm', vllm.__version__, 'torch', torch.__version__)"],
                       capture_output=True, text=True, env=command_environment())
print(check.stdout, check.stderr[-3000:])
assert check.returncode == 0, "Kurulum denetimi başarısız: vllm/torch/matplotlib açılmadı ya da CUDA görünmüyor."

In [ ]:
# 5) Drive denetimi (saniyeler): yazma, üzerine yazma, geri okuma, küçük dosya süresi.
cli("mirror-check", "--mirror-dir", f"{RUN_DRIVE}/mirror")

In [ ]:
# 6) ALFWorld verisi (~2-5 dk): manifestlerin kurulduğu sürüm 0.2.2, sha256 ile doğrulanır.
# (`alfworld-download` başka bir sürüm indirir; onunla hiçbir oyun açılmaz.) Ardından üç
# manifestin (bölmeler, pilot, fayda bankası) 1.490 oyununun her biri hash'iyle denetlenir;
# tutmazsa model indirilmeden durur.
import os
os.environ["ALFWORLD_DATA"] = "/content/alfworld-data"
DATA_ROOT = "/content/alfworld-data/json_2.1.1"
cli("fetch-data", "--target", "/content/alfworld-data")
cli("verify-data", "--data-root", DATA_ROOT, "--repo", REPO)

In [ ]:
# 6b) Koşu provası bu makineyi de kapsıyor mu: aynı commit, Python, sabit sürümler, model
# revizyonu, tokenizer, protokol ve worker sayısı? Kapsamıyorsa model indirilmeden durulur ve GPU
# bırakılır: koşu provası saatler sürer, A100'de yinelenmez. (`run` aynı denetimi kendisi de yapar.)
try:
    cli("rehearsal-check", "--kind", "run", "--passed", PASSED, "--repo", REPO, "--model", MODEL,
        "--revision", MODEL_REVISION, "--workers", WORKERS)
except RuntimeError:
    print("Hata worker süreçleriyle ilgiliyse (yukarıda \"replaced every\"): WORKERS en az 2 "
          "ve Python 3.11 ya da sonrası gerekir; prova da aynısını ister.")
    print("Yoksa bu makine ya da ayarlar koşu provasındakinden farklı (fark yukarıda). "
          "EXP-0001-run-rehearsal.ipynb'yi CPU çalışma zamanında CODE_REF = \"" + COMMIT
          + "\" (" + ("koşunun kendi commit'i" if run_commit else "bu commit") + "), MODEL = \""
          + MODEL + "\" ve WORKERS = " + str(WORKERS) + " ile yeniden çalıştırın; sonra bu "
          "notebook'u yeniden başlatın.")
    raise


In [ ]:
# 7) Aynı A100 kontrolü, deneyin kendi koduyla (fail-closed).
cli("gpu-check")

In [ ]:
# 8) Model: pilotun sabitlediği revizyon (koşu başında kaydedilen ağırlık hash'iyle karşılaştırılır).
cli("prepare-model", "--model", MODEL, "--revision", MODEL_REVISION,
    "--out", "/content/prepared.json")

In [ ]:
# 9) Model sunucusu (~5-15 dk): vllm serve, yalnızca metin, düşünme kapalı; batch-invariant
# çekirdekler kapalı, çünkü vLLM'de Qwen3.8'in GDN katmanları için yoklar (ADR-0011 D2).
# Hazır olunca "the model server ... serves" yazar; günlük: /content/vllm-server.log
# Sunucu açılmazsa günlüğün sonu hata çıktısında görünür; tamamı Drive'da logs/ altına kopyalanır.
cli("serve", "--prepared", "/content/prepared.json", "--log", "/content/vllm-server.log",
    "--info", "/content/vllm-server.json")

In [ ]:
# 10) Ön kontrol (3 model çağrısı, hiçbir deney kaydına girmez): G0'ın kendi mesaj biçimiyle
# (sistem + kullanıcı) ReAct biçimli bir yanıt, aynısı bir geçit bölümü gibi (sıcaklık 0 ve bir
# madencilik tohumu), ve öğretmeninkiyle (tek kullanıcı mesajı) miner'ın JSON şemasına uyan bir
# yanıt. Miner normalde ancak 540 lifetime bölümünden sonra, geçit ondan da sonra ilk kez
# çalışır; burada hata verirse lifetime'a CU harcamadan durursunuz.
cli("preflight", "--prepared", "/content/prepared.json", "--gate-shape")


In [ ]:
# 11) EXP-0001: başlat ya da devam et. Bu hücre saatlerce sürebilir; sekmeyi açık tutun.
# Her 50 bölümde bir ilerleme satırı yazar.
cli("run", "--prepared", "/content/prepared.json", "--data-root", DATA_ROOT, "--repo", REPO,
    "--server-info", "/content/vllm-server.json", "--workers", WORKERS,
    "--local-dir", "/content/exp0001-run", "--mirror-dir", f"{RUN_DRIVE}/mirror", "--run-id", RUN_ID,
    "--rehearsal-passed", PASSED,
    *([] if STOP_AFTER_HOURS is None else ["--stop-after-hours", STOP_AFTER_HOURS]))

In [ ]:
# 12) Durum (yalnızca okur): bölümler, çağrılar, çalışılan saat, oturumlar.
cli("status", "--run-dir", "/content/exp0001-run", "--mirror-dir", f"{RUN_DRIVE}/mirror")

In [ ]:
# 13) Koşu bittiyse rapor Drive'a kopyalanır.
import json, shutil
if os.path.exists("/content/exp0001-run/report.json"):
    shutil.copy("/content/exp0001-run/report.json", f"{RUN_DRIVE}/report.json")
    report = json.load(open("/content/exp0001-run/report.json"))
    print({k: report.get(k) for k in ("status", "stage", "reason", "episodes", "model_calls")})
    print("Rapor:", f"{RUN_DRIVE}/report.json", "— bu dosyayı gönderin.")
else:
    print("Koşu henüz bitmedi; sonraki oturumda aynı RUN_ID ile 'Tümünü çalıştır' deyin.")

In [ ]:
# 14) Makale grafikleri: rapordan, kapı kayıtlarından ve ledger'dan çizilir; koşu klasörüne ve
# aynaya dokunmaz. Yarım kalan bir koşuda yalnızca o ana kadarki betimsel grafikler çizilir ve
# üstlerinde INCOMPLETE ya da UNFINISHED yazar. PDF'ler makale için, CSV'ler her grafiğin tablo
# görünümü, captions.md açıklamaları ve zorunlu notları taşır. Bitmiş bir koşunun grafiklerini
# sonradan EXP-0001-figures.ipynb (CPU) Drive'daki aynadan yeniden üretir.
import os, shutil  # matplotlib kilitli ortamda (4c denetledi)
from IPython.display import Image, display
FIGURES = "/content/exp0001-figures"
cli("figures", "--run-dir", "/content/exp0001-run", "--out-dir", FIGURES)
shutil.rmtree(f"{RUN_DRIVE}/figures", ignore_errors=True)
shutil.copytree(FIGURES, f"{RUN_DRIVE}/figures")
for name in sorted(os.listdir(FIGURES)):
    if name.endswith(".png"):
        display(Image(filename=f"{FIGURES}/{name}"))
print("Grafikler:", f"{RUN_DRIVE}/figures")

In [ ]:
# Son) Drive'ı boşalt ve GPU'yu bırak: bağlı kaldığı her saat ~7.5 CU harcar. İncelemek
# isterseniz AUTO_RELEASE = False yapın. (Bir hücre hata verirse 0. hücre aynısını yapar.)
if AUTO_RELEASE:
    release()

**Bitti mi?** `report.json` dosyasını gönderin. Makale grafikleri her oturumun sonunda `heritagent/runs/<RUN_ID>/figures/` altına yazılır (PDF, PNG, CSV tablo görünümü ve `captions.md`); bitmiş bir koşununkileri `EXP-0001-figures.ipynb` CPU'da yeniden üretir. Final istatistik koşu sonrası Critic incelemesinden geçer; hiçbir sonuç "doğrulandı" diye etiketlenmez (SPEC §32).